# Módulo 06 — Juegos Extensivos e Inducción hacia Atrás

**Objetivos**: Representar juegos secuenciales con `networkx`. Implementar backward induction para encontrar el SPE. Demostrar por qué la amenaza de guerra de precios no es creíble.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
print('NetworkX', nx.__version__)

## 1. Representar el árbol de juego

Usamos un DiGraph de networkx. Cada nodo tiene atributos:
- `player`: quién decide (None si es hoja)
- `payoffs`: (u1, u2) si es hoja
- `label`: nombre de la acción

In [ ]:
def build_entry_game():
    """Juego de entrada al mercado: Entrante vs Incumbente."""
    G = nx.DiGraph()
    G.add_node('root',  player=1, label='Entrante')
    G.add_node('no',    player=None, payoffs=(0, 4), label='No entrar')
    G.add_node('in',    player=2, label='Incumbente')
    G.add_node('war',   player=None, payoffs=(-1, -1), label='Guerra')
    G.add_node('acc',   player=None, payoffs=(1, 2), label='Acomodar')

    G.add_edge('root', 'no',  action='No entrar')
    G.add_edge('root', 'in',  action='Entrar')
    G.add_edge('in',   'war', action='Guerra')
    G.add_edge('in',   'acc', action='Acomodar')
    return G

G_entry = build_entry_game()
print('Nodos:', list(G_entry.nodes(data=True))[:2])

## 2. Backward Induction recursivo

In [ ]:
def backward_induction(G, node):
    """
    Resuelve el subjuego con raíz en `node` por inducción hacia atrás.
    Devuelve: (payoffs_opt, optimal_path)
      payoffs_opt: (u1, u2) del resultado óptimo desde este nodo
      optimal_path: lista de aristas (u, v) que forman el SPE
    """
    data = G.nodes[node]

    # Nodo terminal
    if data['player'] is None:
        return data['payoffs'], []

    player_idx = data['player'] - 1  # índice 0 o 1
    best_payoffs = None
    best_edge = None
    best_path = []

    for child in G.successors(node):
        payoffs, path = backward_induction(G, child)
        if best_payoffs is None or payoffs[player_idx] > best_payoffs[player_idx]:
            best_payoffs = payoffs
            best_edge = (node, child)
            best_path = path

    return best_payoffs, [best_edge] + best_path

payoffs, spe_path = backward_induction(G_entry, 'root')
print(f'SPE payoffs: {payoffs}')
print(f'SPE path (aristas): {spe_path}')

In [ ]:
# Visualización del árbol con SPE resaltado
def draw_game_tree(G, spe_path, title=''):
    pos = nx.nx_agraph.graphviz_layout(G, prog='dot') if hasattr(nx.nx_agraph, 'graphviz_layout') else nx.spring_layout(G)
    # Fallback layout si no hay graphviz
    try:
        pos = nx.nx_agraph.graphviz_layout(G, prog='dot')
    except Exception:
        pos = nx.drawing.nx_pydot.pydot_layout(G, prog='dot') if hasattr(nx.drawing, 'nx_pydot') else None
        if pos is None:
            # Manual layout
            pos = {'root': (2,3), 'no':(0,1.5), 'in':(4,1.5), 'war':(3,0), 'acc':(5,0)}

    spe_edges = set(spe_path)
    edge_colors = ['#2d7a50' if e in spe_edges else '#cccccc' for e in G.edges()]
    edge_widths = [3 if e in spe_edges else 1 for e in G.edges()]

    leaves = [n for n, d in G.nodes(data=True) if d['player'] is None]
    internal = [n for n in G.nodes() if n not in leaves]

    plt.figure(figsize=(9, 5))
    nx.draw_networkx_nodes(G, pos, nodelist=internal, node_color='#1a3a5c', node_size=600)
    nx.draw_networkx_nodes(G, pos, nodelist=leaves, node_color='#f5e6d0', node_size=600, edgecolors='#b85c00', linewidths=2)
    nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=edge_widths, arrows=True, arrowsize=20, min_source_margin=20, min_target_margin=20)

    # Etiquetas de nodos
    labels = {}
    for n, d in G.nodes(data=True):
        if d['player'] is not None:
            labels[n] = f'J{d["player"]}\n{d["label"]}'
        else:
            labels[n] = f'{d["payoffs"]}'
    nx.draw_networkx_labels(G, pos, labels, font_size=8, font_color='white')
    for n in leaves:
        plt.annotate(str(G.nodes[n]['payoffs']), xy=pos[n], ha='center', va='center', fontsize=9, color='#1a3a5c', fontweight='bold')

    # Etiquetas de aristas
    edge_labels = nx.get_edge_attributes(G, 'action')
    nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=8, font_color='#b85c00')

    plt.title(title, fontweight='bold')
    plt.axis('off'); plt.tight_layout(); plt.show()

draw_game_tree(G_entry, spe_path, 'Entrada al Mercado — SPE en verde')

## 3. La amenaza de guerra no es creíble

El incumbente amenaza con iniciar una guerra de precios si el entrante entra. Pero la guerra (−1, −1) es peor que acomodar (1, 2) para el incumbente. El rival lo sabe → la amenaza es vacía → el SPE es (Entrar, Acomodar).

In [ ]:
# Verificar: ¿cuándo sería creíble la amenaza?
print('Payoffs si incumbente hace guerra:   (-1, -1)')
print('Payoffs si incumbente acomoda:        (1,  2)')
print('\n¿Prefiere el incumbente (J2) la guerra?', -1 > 2, '→ NO, la amenaza no es creíble.')
print('\nLa amenaza sería creíble solo si el coste de la guerra fuera')
print('menor que el coste de la acomodación para el incumbente.')

## 4. Juego del Ultimátum

J1 reparte 10 unidades: propone (7, 3) o (5, 5). J2 puede aceptar o rechazar. Backward induction: J2 acepta cualquier oferta > 0.

In [ ]:
G_ult = nx.DiGraph()
G_ult.add_node('root', player=1, label='J1')
for offer, payoff in [('7_3', (7,3)), ('5_5', (5,5))]:
    G_ult.add_node(f'prop_{offer}', player=2, label=f'J2 recibe {payoff[1]}')
    G_ult.add_node(f'acc_{offer}',  player=None, payoffs=payoff, label='Aceptar')
    G_ult.add_node(f'rej_{offer}',  player=None, payoffs=(0,0), label='Rechazar')
    G_ult.add_edge('root', f'prop_{offer}', action=f'({payoff[0]},{payoff[1]})')
    G_ult.add_edge(f'prop_{offer}', f'acc_{offer}', action='Aceptar')
    G_ult.add_edge(f'prop_{offer}', f'rej_{offer}', action='Rechazar')

payoffs_ult, spe_ult = backward_induction(G_ult, 'root')
print(f'SPE del Ultimátum: {payoffs_ult}')
print('Inducción hacia atrás predice que J2 acepta cualquier oferta positiva.')
print('Resultado: J1 propone (7,3) y J2 acepta → (7, 3)')

## Ejercicios

**Ejercicio 1**: Modifica el Juego de Entrada para que el coste de la guerra sea −0.5 para el incumbente (en lugar de −1). ¿Cambia el SPE?

**Ejercicio 2**: Implementa el Juego de Confianza (Trust Game): J1 puede confiar (su dotación 4 se triplica a 12 para J2) o no confiar. Si confía, J2 puede devolver la mitad o quedarse todo. Resuelve con backward induction.

In [ ]:
# Tu código aquí
